In [2]:
import pandas as pd
df = pd.read_csv('train_test_network.csv', encoding='utf-8')
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 211043 entries, 0 to 211042
Data columns (total 44 columns):
 #   Column                  Non-Null Count   Dtype  
---  ------                  --------------   -----  
 0   src_ip                  211043 non-null  object 
 1   src_port                211043 non-null  int64  
 2   dst_ip                  211043 non-null  object 
 3   dst_port                211043 non-null  int64  
 4   proto                   211043 non-null  object 
 5   service                 211043 non-null  object 
 6   duration                211043 non-null  float64
 7   src_bytes               211043 non-null  int64  
 8   dst_bytes               211043 non-null  int64  
 9   conn_state              211043 non-null  object 
 10  missed_bytes            211043 non-null  int64  
 11  src_pkts                211043 non-null  int64  
 12  src_ip_bytes            211043 non-null  int64  
 13  dst_pkts                211043 non-null  int64  
 14  dst_ip_bytes        

In [3]:
import pandas as pd
import numpy as np
import pennylane as qml
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report, roc_curve, auc
from sklearn.utils.class_weight import compute_class_weight
from imblearn.over_sampling import SMOTE
from scipy.stats import mode
import time
import os
from joblib import dump, load
import os
from scipy.stats import mode
from tqdm import tqdm
from sklearn.decomposition import PCA
import torch
import torch.nn as nn
import torch.optim as optim
    

SEED = 42
NUM_FEATURES = 13

In [4]:
categorical_cols = df.select_dtypes(include='object').columns
label_encoders = {}

le = LabelEncoder()
for col in categorical_cols:
    df[col] = le.fit_transform(df[col].astype(str))
    label_encoders[col] = le
    print(f"\nColumn: {col}")
    print(dict(zip(le.classes_, le.transform(le.classes_))))

feature_names = df.columns

X = df.drop('label', axis=1)
y = df['label']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=SEED)
print(f"Original class distribution: {np.bincount(y_train)}")

scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)


pca = PCA(n_components=NUM_FEATURES, random_state=SEED)
X_train_pca = pca.fit_transform(X_train)
X_test_pca = pca.transform(X_test_scaled)

print("Features Selected: ")
for i, component in enumerate(pca.components_):
    # Get the index of the feature with the largest absolute value in this component
    important_idx = np.argmax(np.abs(component))
    print(f"Principal Component {i+1}: most important feature is '{feature_names[important_idx]}'")


X_train = X_train_pca
X_test = X_test_pca

print(X_train.shape)
print(X_test.shape)



Column: src_ip
{'13.55.50.68': 0, '172.17.0.5': 1, '172.17.0.7': 2, '172.17.0.8': 3, '192.168.1.1': 4, '192.168.1.103': 5, '192.168.1.133': 6, '192.168.1.152': 7, '192.168.1.169': 8, '192.168.1.17': 9, '192.168.1.180': 10, '192.168.1.184': 11, '192.168.1.186': 12, '192.168.1.190': 13, '192.168.1.191': 14, '192.168.1.192': 15, '192.168.1.193': 16, '192.168.1.194': 17, '192.168.1.195': 18, '192.168.1.250': 19, '192.168.1.30': 20, '192.168.1.31': 21, '192.168.1.32': 22, '192.168.1.33': 23, '192.168.1.34': 24, '192.168.1.36': 25, '192.168.1.37': 26, '192.168.1.38': 27, '192.168.1.39': 28, '192.168.1.46': 29, '192.168.1.49': 30, '192.168.1.6': 31, '192.168.1.79': 32, '199.115.158.193': 33, '203.14.129.10': 34, '220.158.215.20': 35, '2405:6e00:10ce:2c00:20c:29ff:fed2:b002': 36, '2405:6e00:10ce:2c00:20c:29ff:feee:e07a': 37, '2405:6e00:10ce:2c00:8520:4eba:6770:702a': 38, '2405:6e00:10ce:2c00:9064:17:85d2:57d4': 39, '2405:6e00:10ce:2c00:c058:dbaa:f212:a2d8': 40, 'fe80::1087:c341:7263:ead3': 41

In [5]:
def create_sequential_batches(X_train, y_train, X_test, y_test, train_batch_size=300, test_batch_size=100):
    train_batches = []
    test_batches = []
    num_batches = min(len(X_train) // train_batch_size, len(X_test) // test_batch_size)
    for i in range(num_batches):
        train_start = i * train_batch_size
        train_end = train_start + train_batch_size
        test_start = i * test_batch_size
        test_end = test_start + test_batch_size
        train_batches.append((X_train[train_start:train_end], y_train[train_start:train_end]))
        test_batches.append((X_test[test_start:test_end], y_test[test_start:test_end]))
    return train_batches, test_batches

# Usage:
train_batches, test_batches = create_sequential_batches(X_train, y_train, X_test, y_test)

In [6]:
SEED = 42
TRAIN_BATCH_SIZE = 300
TEST_BATCH_SIZE = 100
NUM_BATCHES = 10

from joblib import Parallel, delayed


dev = qml.device("default.qubit", wires=NUM_FEATURES)

@qml.qnode(dev)
def kernel_circuit(x1, x2):
    qml.AngleEmbedding(x1, wires=range(NUM_FEATURES))
    qml.adjoint(qml.AngleEmbedding)(x2, wires=range(NUM_FEATURES))
    return qml.probs(wires=range(NUM_FEATURES))

def kernel(x1, x2):
    return kernel_circuit(x1, x2)

def compute_kernel_matrix(X1, X2):
    n_samples_1, n_samples_2 = len(X1), len(X2)
    kernel_matrix = np.zeros((n_samples_1, n_samples_2))
    for i in range(n_samples_1):
        for j in range(n_samples_2):
            probs = kernel_circuit(X1[i], X2[j])
            kernel_matrix[i, j] = probs[0]  # this represents the inner product now
    return kernel_matrix

In [7]:
from sklearn.model_selection import GridSearchCV

# Store metrics
all_test_preds = []
all_test_labels = []
train_accuracies = []
test_accuracies = []


for batch_idx in range(10):
    print(f"\nBatch {batch_idx + 1}/10")

    # Get batch data
    X_batch_train, y_batch_train = train_batches[batch_idx]
    X_batch_test, y_batch_test = test_batches[batch_idx]

    print("- Computing training kernel...")
    K_train = compute_kernel_matrix(X_batch_train, X_batch_train)

    print("- Computing testing kernel...")
    K_test = compute_kernel_matrix(X_batch_test, X_batch_train)

    if batch_idx == 4:
        K_train_4 = K_train
        K_test_4 = K_test

    # Hyperparameter tuning for C
    print("- Tuning C value...")
    param_grid = {"C": [0.1, 1, 10, 20, 30, 40, 50, 60]}
    grid = GridSearchCV(SVC(kernel='precomputed'), param_grid, cv=3)
    grid.fit(K_train, y_batch_train)
    best_model = grid.best_estimator_
    print(f"- Best C: {grid.best_params_['C']}")


    # Predictions
    y_pred = best_model.predict(K_test)
    all_test_preds.append(y_pred)
    all_test_labels.append(y_batch_test)

    # Overfitting check
    train_preds = best_model.predict(K_train)
    train_acc = accuracy_score(y_batch_train, train_preds)
    test_acc = accuracy_score(y_batch_test, y_pred)
    train_accuracies.append(train_acc)
    test_accuracies.append(test_acc)

    print(f"- Train Accuracy: {train_acc:.4f}")
    print(f"- Test Accuracy: {test_acc:.4f}")
    if train_acc - test_acc > 0.10:
        print("- Possible overfitting detected (Train >> Test)")



Batch 1/10
- Computing training kernel...
- Computing testing kernel...
- Tuning C value...
- Best C: 10
- Train Accuracy: 0.9967
- Test Accuracy: 0.9800

Batch 2/10
- Computing training kernel...
- Computing testing kernel...
- Tuning C value...
- Best C: 1
- Train Accuracy: 0.9900
- Test Accuracy: 0.9500

Batch 3/10
- Computing training kernel...
- Computing testing kernel...
- Tuning C value...
- Best C: 10
- Train Accuracy: 1.0000
- Test Accuracy: 0.9900

Batch 4/10
- Computing training kernel...
- Computing testing kernel...
- Tuning C value...
- Best C: 1
- Train Accuracy: 0.9933
- Test Accuracy: 0.9900

Batch 5/10
- Computing training kernel...


KeyboardInterrupt: 

In [ ]:
# Final evaluation
all_test_preds = np.concatenate(all_test_preds)
all_test_labels = np.concatenate(all_test_labels)

final_acc = accuracy_score(all_test_labels, all_test_preds)
print(f"\nFinal Batched QSVM Accuracy: {final_acc:.4f}")
print("Classification Report:\n", classification_report(all_test_labels, all_test_preds))

# Confusion matrix
cm = confusion_matrix(all_test_labels, all_test_preds)
plt.figure(figsize=(4, 3))
sns.heatmap(cm, annot=True, fmt='d', cmap="Blues", xticklabels=["Normal", "Anomaly"], yticklabels=["Normal", "Anomaly"], cbar=False, annot_kws={"size": 12})
# plt.title("Confusion Matrix - Ensemble QSVM")
plt.xlabel("Predicted", fontsize=12)
plt.ylabel("True", fontsize=12)
plt.tight_layout()
plt.show()

# Plot train vs test accuracy
plt.plot(range(1, 11), train_accuracies, label="Train Accuracy", marker='o')
plt.plot(range(1, 11), test_accuracies, label="Test Accuracy", marker='s')
plt.title("Train vs Test Accuracy per Batch")
plt.xlabel("Batch #")
plt.ylabel("Accuracy")
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.show()


In [ ]:
n_layers = 7


def qnn_circuit(inputs, weights):
    qml.AngleEmbedding(inputs, wires=range(NUM_FEATURES))
    qml.StronglyEntanglingLayers(weights, wires=range(NUM_FEATURES))
    return [qml.expval(qml.PauliZ(i)) for i in range(NUM_FEATURES)]

qnode = qml.QNode(qnn_circuit, dev, interface="torch")

class QNNClassifier(nn.Module):
    def __init__(self, n_layers, n_qubits):
        super().__init__()
        weight_shapes = {"weights": (n_layers, n_qubits, 3)}
        self.qlayer = qml.qnn.TorchLayer(qnode, weight_shapes)
        self.fc = nn.Linear(n_qubits, 1)
    
    def forward(self, x):
        x = self.qlayer(x)
        x = self.fc(x)
        return x  # No sigmoid here

In [ ]:
X_train_300 = X_train[:3000]
X_test_100 = X_test[:1000]
y_train_300 = y_train[:3000]
y_test_100 = y_test[:1000]

loss_values = []

# Prepare data
X_train_tensor = torch.tensor(X_train_300, dtype=torch.float32)
y_train_tensor = torch.tensor(y_train_300.values, dtype=torch.float32).unsqueeze(1)
X_test_tensor = torch.tensor(X_test_100, dtype=torch.float32)
y_test_tensor = torch.tensor(y_test_100.values, dtype=torch.float32).unsqueeze(1)

# Use DataLoader for mini-batch training
from torch.utils.data import TensorDataset, DataLoader
train_dataset = TensorDataset(X_train_tensor, y_train_tensor)
batch_size = 32
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)

# Model, loss, optimizer
model = QNNClassifier(n_layers=n_layers, n_qubits=NUM_FEATURES)
criterion = nn.BCEWithLogitsLoss()
optimizer = optim.Adam(model.parameters(), lr=0.01)
print(model.parameters())

# Training loop
epochs = 100
for epoch in range(epochs):
    model.train()
    running_loss = 0.0
    for X_batch, y_batch in train_loader:
        optimizer.zero_grad()
        outputs = model(X_batch)
        loss = criterion(outputs, y_batch)
        loss.backward()
        optimizer.step()
        running_loss += loss.item()
    epoch_loss = running_loss / len(train_loader)
    loss_values.append(epoch_loss)
    if (epoch+1) % 10 == 0:
        print(f"Epoch {epoch+1}/{epochs}, Loss: {running_loss/len(train_loader):.4f}")

# Evaluation
model.eval()
# with torch.no_grad():
#     y_pred_logits = model(X_test_tensor)
#     qnn_preds = (torch.sigmoid(y_pred_logits) > 0.5).int().squeeze().numpy()

with torch.no_grad():
    y_pred_logits = model(X_test_tensor)  # shape: (N, 1)
    qnn_sigmoid = torch.sigmoid(y_pred_logits).squeeze()  # shape: (N,)
    qnn_preds = (qnn_sigmoid > 0.5).int().numpy()

    # Convert to probability distribution over 2 classes: [P(class 0), P(class 1)]
    qnn_probs = torch.stack([1 - qnn_sigmoid, qnn_sigmoid], dim=1).cpu().numpy()  # shape: (N, 2)


accuracy_qnn = accuracy_score(y_test_100, qnn_preds)
print(f"QNN Accuracy on Test Data: {accuracy_qnn:.4f}")

print("Classification Report:\n", classification_report(y_test_100, qnn_preds))

cm = confusion_matrix(y_test_100, qnn_preds)
plt.figure(figsize=(4, 3))
sns.heatmap(cm, annot=True, fmt='d', cmap="Blues", xticklabels=["Normal", "Anomaly"], yticklabels=["Normal", "Anomaly"], cbar=False, annot_kws={"size": 12})
# plt.title("Confusion Matrix - QNN")
plt.xlabel("Predicted", fontsize=12)
plt.ylabel("True", fontsize=12)
plt.tight_layout()
plt.show()

In [ ]:
qnn_model = model
qsvm_preds = all_test_preds
X_meta = np.vstack([qsvm_preds, qnn_preds]).T  # shape: (n_samples, 2)
from sklearn.ensemble import RandomForestClassifier

# Initialize and train the meta-model
meta_model_rf = RandomForestClassifier(n_estimators=100, random_state=42)
meta_model_rf.fit(X_meta, y_test[:1000])  # match y_test slicing
ensemble_rf_preds = meta_model_rf.predict(X_meta)

print("Random Forest Meta-Model Accuracy:", accuracy_score(y_test[:1000], ensemble_rf_preds))
print("\nClassification Report (Random Forest Meta-Model):")
print(classification_report(y_test[:1000], ensemble_rf_preds))


In [ ]:
"""
Complete Noise Robustness Analysis for QSVM and QNN
Uses exact trained models from your notebook - NO RETRAINING
"""

import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import pennylane as qml
import torch
import torch.nn as nn
from sklearn.metrics import accuracy_score, classification_report
from sklearn.svm import SVC
import warnings
warnings.filterwarnings('ignore')

# ============================================================================
# SECTION 1: QNN NOISE ANALYSIS
# ============================================================================

print("="*80)
print("QNN NOISE ROBUSTNESS ANALYSIS")
print("="*80)

# Define noise types and levels
noise_types = ['depolarizing', 'bitflip', 'phaseflip', 'amplitude_damping', 'phase_damping']
noise_levels = [0.0, 0.2, 0.4, 0.6, 0.8, 1.0]

# Results storage
qnn_results = {noise_type: [] for noise_type in noise_types}

# Create noisy device
dev_noisy = qml.device("default.mixed", wires=NUM_FEATURES)

def qnn_circuit_noisy(inputs, weights, noise_type='depolarizing', noise_prob=0.0):
    """
    QNN circuit with noise injection after each variational layer.
    Noise is applied to ALL qubits after each StronglyEntanglingLayer.
    """
    # Initial embedding (noise-free)
    qml.AngleEmbedding(inputs, wires=range(NUM_FEATURES))
    
    # Apply variational layers with noise after each layer
    for layer_idx in range(n_layers):
        # Apply one layer of StronglyEntanglingLayers
        # Each layer contains: Rot gates on all qubits + CNOT entanglers
        for wire in range(NUM_FEATURES):
            qml.Rot(weights[layer_idx, wire, 0], 
                   weights[layer_idx, wire, 1], 
                   weights[layer_idx, wire, 2], 
                   wires=wire)
        
        # Entanglers (CNOTs in ring topology)
        for wire in range(NUM_FEATURES):
            qml.CNOT(wires=[wire, (wire + 1) % NUM_FEATURES])
        
        # INJECT NOISE after this variational layer
        if noise_prob > 0.0:
            for wire in range(NUM_FEATURES):
                if noise_type == 'depolarizing':
                    qml.DepolarizingChannel(noise_prob, wires=wire)
                elif noise_type == 'bitflip':
                    qml.BitFlip(noise_prob, wires=wire)
                elif noise_type == 'phaseflip':
                    qml.PhaseFlip(noise_prob, wires=wire)
                elif noise_type == 'amplitude_damping':
                    qml.AmplitudeDamping(noise_prob, wires=wire)
                elif noise_type == 'phase_damping':
                    qml.PhaseDamping(noise_prob, wires=wire)
    
    # Measurements
    return [qml.expval(qml.PauliZ(i)) for i in range(NUM_FEATURES)]

# Create noisy QNode
qnode_noisy = qml.QNode(qnn_circuit_noisy, dev_noisy, interface="torch")

# Extract trained weights from your model
trained_weights = None
for name, param in qnn_model.named_parameters():
    if 'qlayer' in name and 'weights' in name:
        trained_weights = param.detach().clone()
        print(f"✓ Extracted trained weights: shape {trained_weights.shape}")
        break

if trained_weights is None:
    raise ValueError("Could not find trained weights in qnn_model!")

# Evaluate QNN with different noise types and levels
for noise_type in noise_types:
    print(f"\n{noise_type.upper()} Noise:")
    print("-" * 60)
    
    for noise_prob in noise_levels:
        # Create a temporary model with noisy circuit
        class QNNClassifierNoisy(nn.Module):
            def __init__(self, weights, noise_type, noise_prob):
                super().__init__()
                self.weights = weights
                self.noise_type = noise_type
                self.noise_prob = noise_prob
                self.fc = qnn_model.fc  # Reuse trained linear layer
            
            def forward(self, x):
                batch_size = x.shape[0]
                outputs = []
                for i in range(batch_size):
                    result = qnode_noisy(x[i], self.weights, 
                                        noise_type=self.noise_type, 
                                        noise_prob=self.noise_prob)
                    outputs.append(torch.stack(result))
                x_quantum = torch.stack(outputs)
                x_out = self.fc(x_quantum)
                return x_out
        
        # Create noisy model
        model_noisy = QNNClassifierNoisy(trained_weights, noise_type, noise_prob)
        model_noisy.eval()
        
        # Evaluate on test data
        with torch.no_grad():
            y_pred_logits = model_noisy(X_test_tensor)
            y_pred_probs = torch.sigmoid(y_pred_logits)
            y_pred = (y_pred_probs > 0.5).int().squeeze().numpy()
        
        accuracy = accuracy_score(y_test_100, y_pred)
        qnn_results[noise_type].append(accuracy)
        
        print(f"  Noise Level {noise_prob:.1f}: Accuracy = {accuracy:.4f}")

# Save QNN results
qnn_df = pd.DataFrame(qnn_results, index=noise_levels)
qnn_df.index.name = 'Noise Level'
print("\n" + "="*80)
print("QNN RESULTS SUMMARY")
print("="*80)
print(qnn_df)

# ============================================================================
# SECTION 2: QSVM NOISE ANALYSIS
# ============================================================================

print("\n\n" + "="*80)
print("QSVM NOISE ROBUSTNESS ANALYSIS")
print("="*80)

# Results storage
qsvm_results = {noise_type: [] for noise_type in noise_types}

# We'll use batch 5 data (index 4) as a representative test set
X_qsvm_test = X_test[400:500]  # batch 5 test data
y_qsvm_test = y_test.values[400:500]
X_qsvm_train = X_train[1200:1500]  # batch 5 train data

print(f"QSVM Test Set: {X_qsvm_test.shape}")
print(f"QSVM Train Set (for kernel): {X_qsvm_train.shape}")

# Define noisy kernel circuit
def kernel_circuit_noisy(x1, x2, noise_type='depolarizing', noise_prob=0.0):
    """
    Quantum kernel with noise injection in the feature map.
    Noise is applied after the angle embedding on all qubits.
    """
    # Forward embedding
    qml.AngleEmbedding(x1, wires=range(NUM_FEATURES))
    
    # INJECT NOISE after feature encoding
    if noise_prob > 0.0:
        for wire in range(NUM_FEATURES):
            if noise_type == 'depolarizing':
                qml.DepolarizingChannel(noise_prob, wires=wire)
            elif noise_type == 'bitflip':
                qml.BitFlip(noise_prob, wires=wire)
            elif noise_type == 'phaseflip':
                qml.PhaseFlip(noise_prob, wires=wire)
            elif noise_type == 'amplitude_damping':
                qml.AmplitudeDamping(noise_prob, wires=wire)
            elif noise_type == 'phase_damping':
                qml.PhaseDamping(noise_prob, wires=wire)
    
    # Adjoint embedding
    qml.adjoint(qml.AngleEmbedding)(x2, wires=range(NUM_FEATURES))
    
    # INJECT NOISE after adjoint (same noise)
    if noise_prob > 0.0:
        for wire in range(NUM_FEATURES):
            if noise_type == 'depolarizing':
                qml.DepolarizingChannel(noise_prob, wires=wire)
            elif noise_type == 'bitflip':
                qml.BitFlip(noise_prob, wires=wire)
            elif noise_type == 'phaseflip':
                qml.PhaseFlip(noise_prob, wires=wire)
            elif noise_type == 'amplitude_damping':
                qml.AmplitudeDamping(noise_prob, wires=wire)
            elif noise_type == 'phase_damping':
                qml.PhaseDamping(noise_prob, wires=wire)
    
    return qml.probs(wires=range(NUM_FEATURES))

# Create noisy kernel QNode
dev_kernel_noisy = qml.device("default.mixed", wires=NUM_FEATURES)
kernel_qnode_noisy = qml.QNode(kernel_circuit_noisy, dev_kernel_noisy)

def compute_kernel_matrix_noisy(X1, X2, noise_type, noise_prob):
    """Compute kernel matrix with noisy quantum feature map"""
    n1, n2 = len(X1), len(X2)
    K = np.zeros((n1, n2))
    for i in range(n1):
        for j in range(n2):
            probs = kernel_qnode_noisy(X1[i], X2[j], 
                                       noise_type=noise_type, 
                                       noise_prob=noise_prob)
            K[i, j] = probs[0]  # Inner product
    return K

# Train a fresh SVM on noise-free batch 5 kernel
print("\nTraining reference SVM on noise-free kernel (batch 5)...")
K_train_clean = compute_kernel_matrix(X_qsvm_train, X_qsvm_train)
y_qsvm_train = y_train.values[1200:1500]
svm_batch5 = SVC(kernel='precomputed', C=40)  # Using typical best C value
svm_batch5.fit(K_train_clean, y_qsvm_train)
print("✓ Reference SVM trained")

# Evaluate with different noise types and levels
for noise_type in noise_types:
    print(f"\n{noise_type.upper()} Noise:")
    print("-" * 60)
    
    for noise_prob in noise_levels:
        # Compute noisy kernel matrix
        K_test_noisy = compute_kernel_matrix_noisy(X_qsvm_test, X_qsvm_train, 
                                                   noise_type, noise_prob)
        
        # Predict using trained SVM with noisy kernel
        y_pred = svm_batch5.predict(K_test_noisy)
        accuracy = accuracy_score(y_qsvm_test, y_pred)
        qsvm_results[noise_type].append(accuracy)
        
        print(f"  Noise Level {noise_prob:.1f}: Accuracy = {accuracy:.4f}")

# Save QSVM results
qsvm_df = pd.DataFrame(qsvm_results, index=noise_levels)
qsvm_df.index.name = 'Noise Level'
print("\n" + "="*80)
print("QSVM RESULTS SUMMARY")
print("="*80)
print(qsvm_df)

# ============================================================================
# SECTION 3: VISUALIZATION
# ============================================================================

print("\n" + "="*80)
print("GENERATING PLOTS")
print("="*80)

# Color scheme
colors = {
    'depolarizing': '#e74c3c',
    'bitflip': '#3498db',
    'phaseflip': '#2ecc71',
    'amplitude_damping': '#f39c12',
    'phase_damping': '#9b59b6'
}

# Plot 1: QNN Noise Robustness
fig, ax = plt.subplots(figsize=(12, 7))
for noise_type in noise_types:
    ax.plot(noise_levels, qnn_results[noise_type], 
           marker='o', linewidth=2.5, markersize=8,
           label=noise_type.replace('_', ' ').title(),
           color=colors[noise_type])

ax.set_xlabel('Noise Probability', fontsize=14, fontweight='bold')
ax.set_ylabel('Accuracy', fontsize=14, fontweight='bold')
ax.set_title('QNN Noise Robustness Analysis', fontsize=16, fontweight='bold', pad=20)
ax.legend(fontsize=11, loc='best', framealpha=0.95)
ax.grid(True, alpha=0.3, linestyle='--')
ax.set_xlim(-0.05, 1.05)
ax.set_ylim(0, 1.05)
plt.tight_layout()
plt.savefig('qnn_noise_analysis.png', dpi=300, bbox_inches='tight')
print("✓ Saved: qnn_noise_analysis.png")
plt.show()

# Plot 2: QSVM Noise Robustness
fig, ax = plt.subplots(figsize=(12, 7))
for noise_type in noise_types:
    ax.plot(noise_levels, qsvm_results[noise_type], 
           marker='s', linewidth=2.5, markersize=8,
           label=noise_type.replace('_', ' ').title(),
           color=colors[noise_type])

ax.set_xlabel('Noise Probability', fontsize=14, fontweight='bold')
ax.set_ylabel('Accuracy', fontsize=14, fontweight='bold')
ax.set_title('QSVM Noise Robustness Analysis', fontsize=16, fontweight='bold', pad=20)
ax.legend(fontsize=11, loc='best', framealpha=0.95)
ax.grid(True, alpha=0.3, linestyle='--')
ax.set_xlim(-0.05, 1.05)
ax.set_ylim(0, 1.05)
plt.tight_layout()
plt.savefig('qsvm_noise_analysis.png', dpi=300, bbox_inches='tight')
print("✓ Saved: qsvm_noise_analysis.png")
plt.show()

# Plot 3: Combined Comparison (Average across noise types)
fig, ax = plt.subplots(figsize=(12, 7))

qnn_avg = np.mean([qnn_results[nt] for nt in noise_types], axis=0)
qsvm_avg = np.mean([qsvm_results[nt] for nt in noise_types], axis=0)

ax.plot(noise_levels, qnn_avg, marker='o', linewidth=3, markersize=10,
       label='QNN (Average)', color='#e74c3c', linestyle='-')
ax.plot(noise_levels, qsvm_avg, marker='s', linewidth=3, markersize=10,
       label='QSVM (Average)', color='#3498db', linestyle='-')

ax.fill_between(noise_levels, qnn_avg, alpha=0.2, color='#e74c3c')
ax.fill_between(noise_levels, qsvm_avg, alpha=0.2, color='#3498db')

ax.set_xlabel('Noise Probability', fontsize=14, fontweight='bold')
ax.set_ylabel('Average Accuracy', fontsize=14, fontweight='bold')
ax.set_title('QNN vs QSVM: Average Noise Robustness Comparison', 
            fontsize=16, fontweight='bold', pad=20)
ax.legend(fontsize=12, loc='best', framealpha=0.95)
ax.grid(True, alpha=0.3, linestyle='--')
ax.set_xlim(-0.05, 1.05)
ax.set_ylim(0, 1.05)
plt.tight_layout()
plt.savefig('qnn_vs_qsvm_noise_comparison.png', dpi=300, bbox_inches='tight')
print("✓ Saved: qnn_vs_qsvm_noise_comparison.png")
plt.show()

# Plot 4: Heatmap comparison
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))

# QNN Heatmap
qnn_matrix = np.array([qnn_results[nt] for nt in noise_types])
im1 = ax1.imshow(qnn_matrix, aspect='auto', cmap='RdYlGn', vmin=0, vmax=1)
ax1.set_xticks(range(len(noise_levels)))
ax1.set_xticklabels([f'{p:.1f}' for p in noise_levels])
ax1.set_yticks(range(len(noise_types)))
ax1.set_yticklabels([nt.replace('_', ' ').title() for nt in noise_types])
ax1.set_xlabel('Noise Probability', fontsize=12, fontweight='bold')
ax1.set_ylabel('Noise Type', fontsize=12, fontweight='bold')
ax1.set_title('QNN Noise Robustness Heatmap', fontsize=14, fontweight='bold', pad=15)
plt.colorbar(im1, ax=ax1, label='Accuracy')

# Add text annotations
for i in range(len(noise_types)):
    for j in range(len(noise_levels)):
        text = ax1.text(j, i, f'{qnn_matrix[i, j]:.2f}',
                       ha="center", va="center", color="black", fontsize=9)

# QSVM Heatmap
qsvm_matrix = np.array([qsvm_results[nt] for nt in noise_types])
im2 = ax2.imshow(qsvm_matrix, aspect='auto', cmap='RdYlGn', vmin=0, vmax=1)
ax2.set_xticks(range(len(noise_levels)))
ax2.set_xticklabels([f'{p:.1f}' for p in noise_levels])
ax2.set_yticks(range(len(noise_types)))
ax2.set_yticklabels([nt.replace('_', ' ').title() for nt in noise_types])
ax2.set_xlabel('Noise Probability', fontsize=12, fontweight='bold')
ax2.set_ylabel('Noise Type', fontsize=12, fontweight='bold')
ax2.set_title('QSVM Noise Robustness Heatmap', fontsize=14, fontweight='bold', pad=15)
plt.colorbar(im2, ax=ax2, label='Accuracy')

# Add text annotations
for i in range(len(noise_types)):
    for j in range(len(noise_levels)):
        text = ax2.text(j, i, f'{qsvm_matrix[i, j]:.2f}',
                       ha="center", va="center", color="black", fontsize=9)

plt.tight_layout()
plt.savefig('noise_robustness_heatmaps.png', dpi=300, bbox_inches='tight')
print("✓ Saved: noise_robustness_heatmaps.png")
plt.show()

# ============================================================================
# SECTION 4: SUMMARY TABLE
# ============================================================================

print("\n" + "="*80)
print("COMPREHENSIVE RESULTS TABLE")
print("="*80)

# Create combined dataframe
results_summary = pd.DataFrame({
    'Model': ['QNN']*len(noise_types) + ['QSVM']*len(noise_types),
    'Noise Type': noise_types + noise_types,
    'p=0.0': [qnn_results[nt][0] for nt in noise_types] + 
             [qsvm_results[nt][0] for nt in noise_types],
    'p=0.2': [qnn_results[nt][1] for nt in noise_types] + 
             [qsvm_results[nt][1] for nt in noise_types],
    'p=0.4': [qnn_results[nt][2] for nt in noise_types] + 
             [qsvm_results[nt][2] for nt in noise_types],
    'p=0.6': [qnn_results[nt][3] for nt in noise_types] + 
             [qsvm_results[nt][3] for nt in noise_types],
    'p=0.8': [qnn_results[nt][4] for nt in noise_types] + 
             [qsvm_results[nt][4] for nt in noise_types],
    'p=1.0': [qnn_results[nt][5] for nt in noise_types] + 
             [qsvm_results[nt][5] for nt in noise_types],
})

print(results_summary.to_string(index=False))

# Save results to CSV
results_summary.to_csv('noise_analysis_results.csv', index=False)
print("\n✓ Saved: noise_analysis_results.csv")

# Calculate degradation rates
print("\n" + "="*80)
print("PERFORMANCE DEGRADATION ANALYSIS")
print("="*80)

for model_name, results in [('QNN', qnn_results), ('QSVM', qsvm_results)]:
    print(f"\n{model_name}:")
    print("-" * 60)
    for noise_type in noise_types:
        clean_acc = results[noise_type][0]
        noisy_acc = results[noise_type][-1]
        degradation = (clean_acc - noisy_acc) / clean_acc * 100
        print(f"  {noise_type.replace('_', ' ').title():20s}: "
              f"{clean_acc:.4f} → {noisy_acc:.4f} "
              f"(degradation: {degradation:.1f}%)")

print("\n" + "="*80)
print("NOISE ANALYSIS COMPLETE")
print("="*80)
print("\nGenerated files:")
print("  1. qnn_noise_analysis.png")
print("  2. qsvm_noise_analysis.png")
print("  3. qnn_vs_qsvm_noise_comparison.png")
print("  4. noise_robustness_heatmaps.png")
print("  5. noise_analysis_results.csv")